# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [3]:
# TODO
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print(f'total revenue: ${total_revenue:.2f}')
print(f'total units: {total_units}')
df.head()

total revenue: $8520.00
total units: 783


,vendor_id,category,qty,price,revenue
0,V-10,Drink,2,24.0,48.0
1,V-18,RainGear,1,12.0,12.0
2,V-18,Drink,3,4.5,13.5
3,V-10,Food,2,12.0,24.0
4,V-18,Drink,3,7.5,22.5


Across the 400 sample orders, total revenue reached $8520.00 on 783 total units sold.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [4]:
# TODO
by_category = (
    df.groupby('category', as_index=False)['revenue']
    .sum()
    .sort_values(by='revenue', ascending=False)
)

by_category['share_pct'] = (by_category['revenue'] / by_category['revenue'].sum()) * 100
by_category

,category,revenue,share_pct
1,Food,4293.0,50.387324
2,Merch,1771.5,20.792254
0,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


The revenue for Food was \$4293 worth 50.3% of the total, for Merch was \$1771.50 worth 20.79% of the total, for Drink was \$1554 worth 18.23% of the total, and for RainGear it was \$901.5 worth 10.58% of total.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [7]:
# TODO
vendor_total = (
    df.groupby('vendor_id')['revenue']
    .agg(avg_order_rev='mean', order_count='count')
    .sort_values(by='avg_order_rev', ascending=False)
)
vendor_total

,avg_order_rev,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


The vendor with the highest average is vendor id 01. It checks out as they have an average order revenue of 22.6 on 94 orders which is pretty close in terns of revenue and count compared to the runner up. v-18

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [10]:
# TODO
merch_share = by_category.loc[by_category['category'] == 'Merch', 'share_pct'].iloc[0]
print(f"Merch accounts for {merch_share:.1f}% of total revenue.")

Merch accounts for 20.8% of total revenue.


Given we've calcuated it previously, merch accounts for 20.8% of the total revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [11]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

print("Row count:", len(joined) == len(df))
print("Revenue total:", abs(joined['revenue'].sum() - df['revenue'].sum()) < 0.01)

unmatched = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()
print("Unmatched vendor ID:", unmatched)

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown / V-18')
# TODO: merge, validate, and report the unmatched vendor

Row count: True
Revenue total: True
Unmatched vendor ID: ['V-18']


**The unmatched vendor, and what I did about it:** Given that there unknown result came when matching the vendor names with the ids, the one ids that was not listed was vendor 18, to keep all transaction data intact without dropping orders, I retained the left join and filled the missing names with 'Unknown / V-18'

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [12]:
# TODO
pivot_report = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

pivot_report

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown / V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [13]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a. For the next game, i would suggest the vendors to pivot their attentions, staff, and work to Food and scaling back on the inventories on the low-margin items. As we can see from the table, Food was highest demanded item, gaining \$4293.00, which constitutes for over 50.4% of the entire \$8,520.00 revenue, compared to like RainGear which raised \$901.50 and covered only about 10.6%. Additionally, vendors like Hoos Burgers should consider bundling drinks with their meals to capture higher beverage revenue

b. Q5/Q6 are the weakest part of the numbers. The vendor join relies on matching order records to vendor identities, and one vendor, V-18, never matched. That's not a minor gap: V-18 was actually our top earner, $2,349.00 across 108 orders, more than any other vendor in the dataset. We had to fall back to labeling it "Unknown / V-18" because there's no nameattached to it in the source data.